# Notebook 01 — BRSET In-Domain Fairness Audit
**BECITHCON 2026 · Fairness under domain shift · Experiment 1 of 4**

This notebook audits subgroup fairness of your trained weighted-BCE EfficientNet-B0 model
on the **BRSET** held-out test set. It produces the formal fairness metrics, confidence
intervals, figures, and a results table for the paper's first results section.

**What it does**
1. Loads the BRSET label file and reconstructs (or loads) your patient-level split.
2. Generates per-label predicted probabilities for the validation and test sets.
3. Computes, per subgroup, per label: AUC, sensitivity and false-positive-rate at a fixed
   screening operating point, and calibration error (ECE).
4. Computes fairness **gaps** (max-min across groups) with **patient-level bootstrap** CIs.
5. Saves three figures and a metrics table (CSV + LaTeX) for the paper.

**Fairness axes:** sex, image quality, camera (primary); age band (secondary, see the
confound note in Section 1 — age is missing for ~91% of NIKON images, so age and camera
are entangled).

**You fill in three paths in the CONFIG cell.** Everything else runs unchanged.


In [ ]:
# ============================== CONFIG ==============================
# Fill these three in. The rest have sensible defaults.

LABELS_CSV   = "labels_brset.csv"          # the file you uploaded
IMAGE_DIR    = "/path/to/brset/images"     # folder containing the fundus images
CHECKPOINT   = "/path/to/weighted_bce.pt"  # your trained FMLDS weighted-BCE model

# Optional. Leave SPLIT_FILE as your FMLDS split to guarantee the SAME test patients.
# If None, the notebook regenerates a patient-level 70/10/20 split and WARNS you.
SPLIT_FILE   = None    # e.g. "brset_split.csv" with columns: patient_id, split

# Optional. If you already have a predictions CSV (image_id + prob_<label> columns)
# set this and inference is skipped. Otherwise leave None and the notebook runs inference.
EXISTING_PREDS = None

OUTPUT_DIR   = "fairness_outputs"
DEVICE       = "cuda"          # "cuda" or "cpu"
SEED         = 42
IMG_SIZE     = 224             # EfficientNet-B0 default
BATCH_SIZE   = 64
NUM_WORKERS  = 4

TARGET_SENSITIVITY = 0.85      # screening operating point; threshold chosen on VAL
N_BOOTSTRAP        = 1000      # patient-level bootstrap reps for gap CIs
MIN_POS_RELIABLE   = 10        # flag any subgroup cell with fewer positives than this
ECE_BINS           = 10


In [ ]:
import os, json, warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
np.random.seed(SEED)
os.makedirs(OUTPUT_DIR, exist_ok=True)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score


## Section 1 — Labels, subgroups, and the patient-level split

We keep the 11 modeled labels (dropping `retinal_detachment`, n=7, and the heterogeneous
`other`), match your FMLDS setup, and derive the subgroup columns.

**Confound note (important, goes in the paper):** `patient_age` is missing for ~33% of
images, and ~91% of those are NIKON-camera images. Age and camera are therefore entangled.
We treat sex, quality, and camera as primary axes and age as a caveated secondary axis.


In [ ]:
# 11 modeled labels (retinal_detachment and 'other' dropped, matching FMLDS)
LABELS = ['diabetic_retinopathy','macular_edema','scar','nevus','amd',
          'vascular_occlusion','hypertensive_retinopathy','drusens','hemorrhage',
          'myopic_fundus','increased_cup_disc']

# Labels with enough positives to headline subgroup analysis (rest are reported but flagged)
HEADLINE_LABELS = ['diabetic_retinopathy','macular_edema','amd','drusens','increased_cup_disc']

df = pd.read_csv(LABELS_CSV)
print("Loaded", df.shape, "| patients:", df['patient_id'].nunique())

# --- subgroup columns ---
# Sex is coded 1/2 in BRSET. We label generically here; confirm the mapping for the paper.
df['sex_group'] = df['patient_sex'].map({1: 'sex_1', 2: 'sex_2'})

# Quality and camera are already categorical
df['quality_group'] = df['quality']                      # Adequate / Inadequate
df['camera_group']  = df['camera']                        # Canon CR / NIKON NF5050

# Age bands (only where age present). Clinically motivated cut points.
def age_band(a):
    if pd.isna(a): return np.nan
    if a < 40:  return '<40'
    if a < 60:  return '40-59'
    return '60+'
df['age_group'] = df['patient_age'].apply(age_band)

print("\nAge present:", df['patient_age'].notna().mean().round(3),
      "| age-missing by camera:")
print(df.assign(m=df['patient_age'].isna()).groupby('camera')['m'].mean().round(3))


In [ ]:
# --- patient-level split ---
def regenerate_split(df, seed=SEED):
    rng = np.random.default_rng(seed)
    pts = df['patient_id'].unique().copy()
    rng.shuffle(pts)
    n = len(pts); n_tr = int(0.70*n); n_va = int(0.10*n)
    tr = set(pts[:n_tr]); va = set(pts[n_tr:n_tr+n_va]); te = set(pts[n_tr+n_va:])
    s = pd.Series('train', index=df['patient_id'])
    s[df['patient_id'].isin(va)] = 'val'
    s[df['patient_id'].isin(te)] = 'test'
    return s.values

if SPLIT_FILE:
    sp = pd.read_csv(SPLIT_FILE)
    key = 'patient_id' if 'patient_id' in sp.columns else 'image_id'
    df = df.merge(sp[[key, 'split']], on=key, how='left')
    assert df['split'].notna().all(), "Some rows did not match the split file."
    print("Loaded split from", SPLIT_FILE)
else:
    df['split'] = regenerate_split(df)
    print("*** WARNING: SPLIT_FILE is None — split was REGENERATED. ***")
    print("*** This MUST match your FMLDS split or the audit leaks training data. ***")
    print("*** Set SPLIT_FILE to your FMLDS split before trusting the numbers.   ***")

# verify zero patient leakage across splits
leak = df.groupby('patient_id')['split'].nunique().max()
assert leak == 1, "Patient leakage across splits!"
print("Split sizes (images):", df['split'].value_counts().to_dict(), "| no patient leakage OK")


## Section 2 — Generate predicted probabilities

If `EXISTING_PREDS` is set, we load it and skip inference. Otherwise we load your
checkpoint and run a forward pass over the val and test images.

**Adapt the model class below to match your FMLDS definition if it differs.** The scaffold
is a torchvision EfficientNet-B0 with an 11-output linear head, which is the standard setup;
if your head or backbone differs, paste your own class here so `load_state_dict` matches.


In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image

class BRSETClassifier(nn.Module):
    # Replace with your FMLDS class if it differs so the checkpoint loads cleanly.
    def __init__(self, n_labels):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        in_f = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(0.2), nn.Linear(in_f, n_labels))
    def forward(self, x):
        return self.backbone(x)

_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
])

class BRSETDataset(Dataset):
    def __init__(self, frame, image_dir):
        self.frame = frame.reset_index(drop=True); self.dir = image_dir
    def __len__(self): return len(self.frame)
    def _find(self, iid):
        for ext in ('', '.jpg', '.jpeg', '.png'):
            p = os.path.join(self.dir, str(iid) + ext)
            if os.path.exists(p): return p
        raise FileNotFoundError(f"image not found for id {iid}")
    def __getitem__(self, i):
        iid = self.frame.loc[i, 'image_id']
        img = Image.open(self._find(iid)).convert('RGB')
        return _tf(img), str(iid)

@torch.no_grad()
def run_inference(frame, model, device):
    loader = DataLoader(BRSETDataset(frame, IMAGE_DIR), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS)
    ids, probs = [], []
    for x, batch_ids in loader:
        p = torch.sigmoid(model(x.to(device))).cpu().numpy()
        probs.append(p); ids.extend(batch_ids)
    probs = np.concatenate(probs, 0)
    out = pd.DataFrame(probs, columns=['prob_'+l for l in LABELS]); out['image_id'] = ids
    return out


In [ ]:
if EXISTING_PREDS:
    preds = pd.read_csv(EXISTING_PREDS)
    print("Loaded predictions from", EXISTING_PREDS, preds.shape)
else:
    device = DEVICE if (DEVICE == 'cpu' or torch.cuda.is_available()) else 'cpu'
    model = BRSETClassifier(len(LABELS)).to(device)
    ckpt = torch.load(CHECKPOINT, map_location=device)
    state = ckpt.get('model_state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state); model.eval()
    print("Model loaded on", device)

    sub = df[df['split'].isin(['val','test'])][['image_id'] + LABELS + ['split']].copy()
    preds = run_inference(sub, model, device)
    preds.to_csv(os.path.join(OUTPUT_DIR, 'brset_val_test_preds.csv'), index=False)
    print("Predictions saved.", preds.shape)

# merge predictions back with labels + subgroups + split
data = df.merge(preds, on='image_id', how='inner')
val  = data[data['split']=='val'].copy()
test = data[data['split']=='test'].copy()
print("val:", val.shape[0], "images | test:", test.shape[0], "images")


## Section 3 — Fairness metric helpers

- **AUC** is threshold-free and reported per subgroup.
- **Sensitivity and FPR** use one fixed threshold per label, chosen on the **validation**
  set to reach the target screening sensitivity, then applied unchanged to the test set.
  Equal sensitivity and equal FPR across groups is the equalized-odds criterion.
- **ECE** measures calibration within a group (are predicted probabilities trustworthy?).


In [ ]:
def safe_auc(y, p):
    y = np.asarray(y); p = np.asarray(p)
    if len(np.unique(y)) < 2: return np.nan
    return roc_auc_score(y, p)

def ece(y, p, n_bins=ECE_BINS):
    y = np.asarray(y); p = np.asarray(p)
    if len(y) == 0: return np.nan
    bins = np.linspace(0, 1, n_bins+1); e = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p > lo) & (p <= hi)
        if m.sum() == 0: continue
        e += (m.mean()) * abs(y[m].mean() - p[m].mean())
    return e

def sens_fpr(y, p, thr):
    y = np.asarray(y); pred = (np.asarray(p) >= thr).astype(int)
    tp = ((pred==1)&(y==1)).sum(); fn = ((pred==0)&(y==1)).sum()
    fp = ((pred==1)&(y==0)).sum(); tn = ((pred==0)&(y==0)).sum()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    fpr  = fp/(fp+tn) if (fp+tn) else np.nan
    return sens, fpr

def threshold_for_sensitivity(y_val, p_val, target):
    # highest threshold whose recall on val is >= target (fewest false positives at that sens)
    y_val = np.asarray(y_val); p_val = np.asarray(p_val)
    if y_val.sum() == 0: return 0.5
    order = np.unique(p_val)[::-1]
    chosen = order[-1]
    for t in order:
        s, _ = sens_fpr(y_val, p_val, t)
        if s >= target: chosen = t; break
    return float(chosen)

# per-label thresholds, fixed on validation
THRESHOLDS = {l: threshold_for_sensitivity(val[l], val['prob_'+l], TARGET_SENSITIVITY)
              for l in LABELS}
print("Per-label thresholds (val-derived):",
      {k: round(v,3) for k,v in THRESHOLDS.items()})


## Section 4 — Subgroup metrics and fairness gaps

For each axis and label we compute per-group metrics on the **test** set and the gap
(max minus min across groups). The positive count behind each cell is recorded so thin,
unreliable cells can be flagged rather than over-interpreted.


In [ ]:
AXES = {'sex': 'sex_group', 'quality': 'quality_group',
        'camera': 'camera_group', 'age': 'age_group'}

def subgroup_table(frame, labels=LABELS):
    rows = []
    for axis, col in AXES.items():
        for g, gdf in frame.dropna(subset=[col]).groupby(col):
            for l in labels:
                y = gdf[l].values; p = gdf['prob_'+l].values
                s, f = sens_fpr(y, p, THRESHOLDS[l])
                rows.append(dict(axis=axis, group=g, label=l, n=len(gdf),
                                 n_pos=int(y.sum()), auc=safe_auc(y,p),
                                 sensitivity=s, fpr=f, ece=ece(y,p)))
    return pd.DataFrame(rows)

metrics = subgroup_table(test)

def gaps(metrics, labels=LABELS):
    rows = []
    for axis in AXES:
        for l in labels:
            sub = metrics[(metrics.axis==axis)&(metrics.label==l)]
            def gap(col):
                v = sub[col].dropna()
                return (v.max()-v.min()) if len(v) >= 2 else np.nan
            rows.append(dict(axis=axis, label=l,
                             auc_gap=gap('auc'), sens_gap=gap('sensitivity'),
                             fpr_gap=gap('fpr'), ece_gap=gap('ece'),
                             min_pos=int(sub['n_pos'].min())))
    return pd.DataFrame(rows)

gap_tbl = gaps(metrics)
print("Headline fairness gaps (test set):")
print(gap_tbl[gap_tbl.label.isin(HEADLINE_LABELS)]
      .round(3).to_string(index=False))


## Section 5 — Patient-level bootstrap confidence intervals

Gaps are recomputed over 1000 resamples of **patients** (not images), giving 95% CIs.
A gap whose CI excludes zero is a disparity we can defend in the paper.


In [ ]:
def bootstrap_gap_ci(frame, axis, label, metric, B=N_BOOTSTRAP, seed=SEED):
    rng = np.random.default_rng(seed)
    col = AXES[axis]; pts = frame['patient_id'].unique()
    by_pt = {pid: idx.values for pid, idx in frame.groupby('patient_id').groups.items()}
    vals = []
    for _ in range(B):
        samp = rng.choice(pts, size=len(pts), replace=True)
        idx = np.concatenate([by_pt[p] for p in samp])
        bf = frame.loc[idx]
        per = []
        for g, gdf in bf.dropna(subset=[col]).groupby(col):
            y = gdf[label].values; p = gdf['prob_'+label].values
            if metric=='auc': per.append(safe_auc(y,p))
            elif metric=='ece': per.append(ece(y,p))
            else:
                s,f = sens_fpr(y,p,THRESHOLDS[label])
                per.append(s if metric=='sensitivity' else f)
        per = [v for v in per if not np.isnan(v)]
        if len(per) >= 2: vals.append(max(per)-min(per))
    if not vals: return (np.nan, np.nan, np.nan)
    return (np.mean(vals), np.percentile(vals,2.5), np.percentile(vals,97.5))

ci_rows = []
for axis in ['sex','quality','camera']:           # primary axes
    for l in HEADLINE_LABELS:
        for metric in ['auc','sensitivity','fpr','ece']:
            m, lo, hi = bootstrap_gap_ci(test, axis, l, metric)
            ci_rows.append(dict(axis=axis, label=l, metric=metric,
                                gap=m, ci_low=lo, ci_high=hi,
                                excludes_zero=(lo>0)))
ci_tbl = pd.DataFrame(ci_rows)
print("Gaps whose 95% CI excludes zero (defensible disparities):")
print(ci_tbl[ci_tbl.excludes_zero].round(3).to_string(index=False))


## Section 6 — Figures for the paper

In [ ]:
# Figure A — subgroup AUC by sex for headline labels
fig, ax = plt.subplots(figsize=(8,4))
sexm = metrics[(metrics.axis=='sex') & (metrics.label.isin(HEADLINE_LABELS))]
piv = sexm.pivot(index='label', columns='group', values='auc').reindex(HEADLINE_LABELS)
piv.plot(kind='bar', ax=ax)
ax.set_ylim(0.5,1.0); ax.set_ylabel('AUC'); ax.set_xlabel('')
ax.set_title('BRSET test AUC by sex'); ax.legend(title='group')
plt.xticks(rotation=30, ha='right'); plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'figA_auc_by_sex.png'), dpi=200); plt.close()

# Figure B — calibration curves by quality for diabetic retinopathy
fig, ax = plt.subplots(figsize=(5,5))
for g, gdf in test.dropna(subset=['quality_group']).groupby('quality_group'):
    y = gdf['diabetic_retinopathy'].values; p = gdf['prob_diabetic_retinopathy'].values
    bins = np.linspace(0,1,11); xs, ys = [], []
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()>0: xs.append(p[m].mean()); ys.append(y[m].mean())
    ax.plot(xs, ys, marker='o', label=g)
ax.plot([0,1],[0,1],'k--',alpha=.5)
ax.set_xlabel('mean predicted prob'); ax.set_ylabel('observed frequency')
ax.set_title('DR calibration by image quality'); ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'figB_calibration_dr_quality.png'), dpi=200); plt.close()

# Figure C — forest plot of sensitivity gaps with bootstrap CIs (sex axis)
fig, ax = plt.subplots(figsize=(7,4))
sub = ci_tbl[(ci_tbl.axis=='sex') & (ci_tbl.metric=='sensitivity')].reset_index(drop=True)
ax.errorbar(sub['gap'], range(len(sub)),
            xerr=[sub['gap']-sub['ci_low'], sub['ci_high']-sub['gap']],
            fmt='o', capsize=4)
ax.axvline(0, color='k', ls='--', alpha=.5)
ax.set_yticks(range(len(sub))); ax.set_yticklabels(sub['label'])
ax.set_xlabel('sensitivity gap (max-min across sex)'); ax.set_title('Sex sensitivity gaps, 95% CI')
plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR,'figC_sens_gap_forest.png'), dpi=200); plt.close()

print("Saved figA_auc_by_sex.png, figB_calibration_dr_quality.png, figC_sens_gap_forest.png to", OUTPUT_DIR)


## Section 7 — Export tables and reliability flags

In [ ]:
metrics.to_csv(os.path.join(OUTPUT_DIR,'subgroup_metrics_full.csv'), index=False)
gap_tbl.to_csv(os.path.join(OUTPUT_DIR,'fairness_gaps.csv'), index=False)
ci_tbl.to_csv(os.path.join(OUTPUT_DIR,'fairness_gaps_ci.csv'), index=False)

# Reliability flags
flagged = metrics[metrics['n_pos'] < MIN_POS_RELIABLE]
print("Subgroup cells with < %d positives (treat as unreliable, do not headline):"
      % MIN_POS_RELIABLE)
print(flagged[['axis','group','label','n_pos']].to_string(index=False))

# LaTeX table of headline sex/quality/camera gaps for the paper
tex = ci_tbl[ci_tbl.metric.isin(['auc','sensitivity','ece'])].copy()
tex = tex[tex.label.isin(HEADLINE_LABELS)]
def fmt(r): return f"{r['gap']:.3f} [{r['ci_low']:.3f}, {r['ci_high']:.3f}]"
tex['cell'] = tex.apply(fmt, axis=1)
wide = tex.pivot_table(index=['axis','label'], columns='metric',
                       values='cell', aggfunc='first')
with open(os.path.join(OUTPUT_DIR,'gaps_table.tex'),'w') as f:
    f.write(wide.to_latex())
print("\nSaved subgroup_metrics_full.csv, fairness_gaps.csv, fairness_gaps_ci.csv, gaps_table.tex")


## Section 8 — Reading the results (and what goes in the paper)

- **Headline the gaps whose CI excludes zero** (Section 5 output). Those are defensible.
- **Sex, quality, camera** are your trustworthy axes. **Age** is reported but caveated
  because of the camera-confounded missingness documented in Section 1; say this plainly
  in the paper rather than hiding it.
- **Calibration gaps (ECE) are the novel angle** your FMLDS paper did not have. If a group
  is well-ranked (good AUC) but poorly calibrated (high ECE), the model's probabilities are
  untrustworthy for that group at deployment, which is a real equity finding.
- The very rare labels flagged in Section 7 should not be headlined; mention them only as
  "underpowered for subgroup analysis."

**Next:** Notebook 02 tests whether group-aware calibration and group-specific thresholds
**close** these gaps (in-domain mitigation), then Notebook 03 repeats the whole audit on
mBRSET to ask whether the disparities survive the domain shift.
